In [7]:
import pandas as pd

# Define the file path
save_path = "D:/embedding/mojin_embeddings.parquet"

# Read the CSV file into a pandas DataFrame
df = pd.read_parquet(save_path)

In [8]:
# Print the column headings
print("Column headings of the merged dataset:")
print(df.columns.tolist())

Column headings of the merged dataset:
['filename', 'session_chunks', 'session_chunks_embeddings']


In [3]:
import os
import boto3
import fitz  # PyMuPDF
from io import BytesIO
import pandas as pd
# Print the column headings
print("Column headings of the merged dataset:")
print(df.columns.tolist())

Column headings of the merged dataset:
['filename', 'text', 'session_chunks', 'session_chunks_tokenized', 'session_chunks_embeddings']


In [10]:
#print(df['session_chunks_embeddings'].iloc[0])
print(type(df['session_chunks_embeddings'].iloc[0]))


<class 'numpy.ndarray'>


In [14]:
import qdrant_client
from qdrant_client.http import models
import numpy as np
import json
import torch  # Import PyTorch

# Initialize Qdrant client
q_client = qdrant_client.QdrantClient("http://localhost:6333")

# Create collection only if it doesn't exist
collection_name = "test_novel2"
if not q_client.collection_exists(collection_name):
    q_client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE)  # Assuming 768-dimensional vectors
    )

# Function to safely convert embeddings into lists
def convert_to_list(value):
    if isinstance(value, list):  
        return value  # Already a valid list
    elif isinstance(value, np.ndarray):  
        return value.tolist()  # Convert NumPy array to list
    elif isinstance(value, torch.Tensor):  
        return value.detach().cpu().tolist()  # Convert PyTorch tensor to list
    elif isinstance(value, str):  
        try:
            return json.loads(value)  # Convert JSON string to list
        except json.JSONDecodeError:
            raise ValueError(f"Invalid JSON string for embeddings: {value}")
    raise ValueError(f"Unsupported type for embeddings: {type(value)}")

# Function to insert data into Qdrant
def insert_data_to_qdrant(merged_df, collection_name):
    for index, row in merged_df.iterrows():
        try:
            vectors = convert_to_list(row['session_chunks_embeddings'])
            chunks = convert_to_list(row['session_chunks'])

            if len(vectors) != len(chunks):
                raise ValueError(f"Row {index}: Mismatch between vectors and chunks (vectors: {len(vectors)}, chunks: {len(chunks)})")

            print(f"Number of vectors in row {index}: {len(vectors)}")

            for i, (vector, chunk) in enumerate(zip(vectors, chunks)):
                payload = {
                    "coordinate": [index, i],
                    "chunk": chunk
                }
                vector_data = models.PointStruct(
                    id=index * 1000 + i,  
                    vector=vector,
                    payload=payload
                )
                q_client.upsert(
                    collection_name=collection_name,
                    points=[vector_data]
                )

            print(f"Inserted vectors and chunks for row {index} into Qdrant.")
        except Exception as e:
            print(f"Skipping row {index} due to error: {e}")

# Run insertion function
insert_data_to_qdrant(df, collection_name)

print("Data insertion complete!")


Number of vectors in row 0: 302
Inserted vectors and chunks for row 0 into Qdrant.
Number of vectors in row 1: 325
Inserted vectors and chunks for row 1 into Qdrant.
Number of vectors in row 2: 296
Inserted vectors and chunks for row 2 into Qdrant.
Number of vectors in row 3: 305
Inserted vectors and chunks for row 3 into Qdrant.
Number of vectors in row 4: 328
Inserted vectors and chunks for row 4 into Qdrant.
Number of vectors in row 5: 316
Inserted vectors and chunks for row 5 into Qdrant.
Number of vectors in row 6: 352
Inserted vectors and chunks for row 6 into Qdrant.
Number of vectors in row 7: 230
Inserted vectors and chunks for row 7 into Qdrant.
Number of vectors in row 8: 309
Inserted vectors and chunks for row 8 into Qdrant.
Number of vectors in row 9: 319
Inserted vectors and chunks for row 9 into Qdrant.
Number of vectors in row 10: 321
Inserted vectors and chunks for row 10 into Qdrant.
Number of vectors in row 11: 232
Inserted vectors and chunks for row 11 into Qdrant.


In [4]:
import qdrant_client
from qdrant_client.http import models
import numpy as np
import json
import torch  # Import PyTorch

In [5]:
# Initialize Qdrant client
q_client = qdrant_client.QdrantClient("http://localhost:6333")

# Create collection only if it doesn't exist
collection_name = "test_novel2"

In [6]:
from typing import List, Dict

def search_vector(q_client, embedded_vector: List[float], collection_name: str, top_k: int = 5) -> List[Dict[str, any]]:
    """
    Perform a search in Qdrant using an embedded vector and return the chunks and coordinates.
    
    :param q_client: The initialized Qdrant client.
    :param embedded_vector: The vector to search for.
    :param collection_name: The name of the Qdrant collection to search within.
    :param top_k: The number of nearest vectors to retrieve.
    :return: List of dictionaries containing chunks and their corresponding coordinates.
    """
    print("Performing search...")

    # Perform search using Qdrant
    search_result = q_client.search(
        collection_name=collection_name,
        query_vector=embedded_vector,
        limit=top_k
    )
    
    if search_result:
        # Collect the chunks and coordinates of the nearest vectors
        results = [
            {
                "chunk": result.payload['chunk'],
                "coordinate": result.payload['coordinate']
            }
            for result in search_result
        ]
        return results
    else:
        print("No search results found.")
        return []

# Example usage:
# q_client = initialize_your_qdrant_client()
# results = search_vector(q_client, your_embedded_vector, "vector_collection")
# for result in results:
#     print(f"Chunk: {result['chunk']}, Coordinate: {result['coordinate']}")

In [7]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
def get_local_prompt_embedding(prompt):
    """
    Generates an embedding for a single prompt string using the local M3e-Small embedding model.
    
    Args:
        prompt (str): The text prompt to embed.
    
    Returns:
        torch.Tensor: The embedding tensor for the given prompt.
    """
    embedding = embedding_model.encode(prompt, convert_to_tensor=True)
    return embedding


C:\Users\zhuqi\Documents\trading_bot\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Load BERT-based tokenizer & embedding model
tokenizer = AutoTokenizer.from_pretrained("moka-ai/m3e-small")
embedding_model = SentenceTransformer("moka-ai/m3e-small")

In [29]:
prompt2 = "右方之火能用的魔法都有哪些"
prompt = 'what is the annuity payment plan'
prompt2_embedding = get_local_prompt_embedding(prompt2)

In [30]:
results = search_vector(q_client, prompt2_embedding, "test_novel2")
for result in results:
    print(f"Chunk: {result['chunk']}, Coordinate: {result['coordinate']}")

Performing search...
Chunk: 」
「能够知道右方之火现在去了哪里吗？」
或许坐上救护车会好一点，不过艾莉莎莉娜拒绝了。大概因为和右方之火的战斗已经使魔法暴露在了民众面前了，事后觉得有点内疚吧。或许是要避免以一种麻烦局面从现场撤退吧。不过这个也只有艾莉莎莉娜本人清楚了。
「……大概，是国境那一边的那个基地吧。」
上条稍许考虑了一下以后，回答了艾莉莎莉娜的问题。
「原本，右方之火就是在那里进行着某种准备的。还专门把那里居住的人们都强制赶走了。很可能是要把莎夏带到那里，然后用她做些什么吧。」
右方之火到底要做什么，目前还难以判断。
但是，右方之火光是做「预先准备」就已经造成了这么大的损害了，从第三次世界大战大概也包括在其中吧。这样考虑下来，右方之火接下来要做的事，很可能还要「在此之上」。不管怎么样，不能坐视不理，不能再让他引发更糟糕的事态。
「我会想办法的。」
稍微想了下以后，上条对艾莉莎莉娜这样说道。
「我会想点办法对付那, Coordinate: [19, 220]
Chunk: 到据点的右方之火，正在于俄罗斯成教的司教，尼古兰·托尔斯泰进行魔法通信。
<连无人兵器都出动的学园都市，战力是压倒性的。听信你的花言巧语盲目推进战争的我们，正向着什么样的末路前进我终于明白了啊！！>
「不用担心。大天使·神之力（加百列）。有这个最终武器的话，你还能说出这种梦话么？」
（……本来，那东西也不是为了这么无聊的事才拿到手的呢）
由另一种法则产生的天使，学园都市的AIM扩散力场集合体，风斩冰华抖动着背上的翅膀，穿越日本海的上空。
理由只有一个，为了帮助她的「朋友们」。
「请不要，对我的「朋友们」出手……出手的话，就算是同归于尽，我也要与你为敌。」
位于英国首都的圣乔治大教堂内，史提尔·马格努斯愤怒异常。他的眼前，一个瘦小的人影以不自然的举动缓缓坐了起来。
脑中记忆保管着10万3000本魔导书的少女，茵蒂克丝。
「——敌对行，确认。现在开始……解析敌人术式，以及……执行对应的特定魔法, Coordinate: [20, 9]
Chunk: 说的话，如同疑问句的形式般喷向右方之火。
「为什么要把「伯利恒之星」弄到这么巨大呢？这里恐怕是你这混蛋为了魔法能够安全而切实地施展出来而准备的仪式场吧。不过呢，如果「右方之火」真的是最强的存在的话，又有什么必要特意从全世界的

In [32]:
!pip install azure-ai-inference

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple



[notice] A new release of pip is available: 22.2.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import torch
from sentence_transformers import SentenceTransformer
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential


AZURE_ENDPOINT = "https://DeepSeek-V3-xqjkp.eastus2.models.ai.azure.com"
AZURE_API_KEY = "zT7QmBLIeeILTQmuG5bnnwsgkFOC8gfw"
client = ChatCompletionsClient(
    endpoint=AZURE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_API_KEY),
)





In [10]:
# === HyDE Prompt Template ===
HYPOTHETICAL_ANSWER_TEMPLATE = """
你是一位小说创作助理，你的任务是根据提出的问题，结合小说的写作风格，生成一段合理的、详细的、风格一致的假设性文本，以便用于语义搜索。

请遵循以下要求：
1. 模拟小说中真实段落的写作风格。
2. 回答中应包含足够细节，以便向量检索系统能够匹配到相似的真实内容。
3. 如有提供的背景片段，请参考其语言风格与表达方式。
4. 避免直接重复问题内容，而是通过合理推理与想象，生成与之高度相关的自然语言段落。

【问题】
{text}

{background_section}

【请输出模拟的小说段落】
"""

In [11]:
# === Template Formatter ===
def build_hypothetical_prompt(user_prompt, background_chunks=None):
    background_text = ""
    if background_chunks:
        joined_background = "\n".join(background_chunks)
        background_text = f"\n【背景片段】\n{joined_background}"

    return HYPOTHETICAL_ANSWER_TEMPLATE.replace("{text}", user_prompt).replace("{background_section}", background_text)

# === Use Azure ChatCompletionsClient to Generate Hypothetical Answer ===
def convert_text_to_json(text):
    try:
        response = client.complete(
            messages=[
                {"role": "system", "content": "你是一位中文小说创作代理。请根据输入，模拟输出小说风格的段落。"},
                {"role": "user", "content": text}
            ]
        )
        raw_output = response.choices[0].message.content.strip()
        print("\n📘 Hypothetical Answer Generated:\n", raw_output)
    except Exception as e:
        raw_output = f"Error: {str(e)}"
    
    return raw_output


In [12]:
def generate_hypothetical_query(prompt, q_client, top_k_for_context=2):
    print("🔍 Step 1: Retrieve Initial Chunks for Hypothetical Generation")

    prompt_embedding = embedding_model.encode(prompt, convert_to_tensor=True)
    initial_results = search_vector(q_client, prompt_embedding, "test_novel2", top_k=top_k_for_context)

    for i, result in enumerate(initial_results):
        print(f"\nInitial Chunk {i+1}:\n{result['chunk']}")
        print(f"Coordinate: {result['coordinate']}")

    supporting_text = [res["chunk"] for res in initial_results]
    full_prompt = build_hypothetical_prompt(prompt, supporting_text)

    print("\n🧠 Step 2: Generating Hypothetical Answer with Azure")
    hypothetical_answer = convert_text_to_json(full_prompt)

    return hypothetical_answer


In [13]:
def search_vector_with_neighbors(
    q_client,
    embedded_vector: List[float],
    collection_name: str,
    top_k: int = 5
) -> List[Dict[str, any]]:
    """
    Search using vector and return each hit's chunk + its valid neighbors.
    Handles 2D coordinate like [row_index, chunk_index], staying within the same row.
    Adds print statements for debugging.
    """
    print("🔍 Step 1: Vector Search")
    search_results = q_client.search(
        collection_name=collection_name,
        query_vector=embedded_vector,
        limit=top_k
    )

    all_coords_to_fetch = set()
    primary_coords = []

    for result in search_results:
        coord = result.payload.get("coordinate")
        print(f"📎 Raw coordinate: {coord}")

        if not (isinstance(coord, list) and len(coord) == 2 and all(isinstance(x, int) for x in coord)):
            print("   ⚠️ Skipping invalid coordinate:", coord)
            continue

        row_id, chunk_id = coord
        primary_coords.append((row_id, chunk_id))
        all_coords_to_fetch.update([
            (row_id, chunk_id - 1),
            (row_id, chunk_id),
            (row_id, chunk_id + 1)
        ])

    print(f"✅ Primary coords: {primary_coords}")
    print(f"📌 All needed coords (with neighbors): {sorted(all_coords_to_fetch)}")

    print("📦 Step 2: Fetch all points from Qdrant")
    points, _ = q_client.scroll(
        collection_name=collection_name,
        with_payload=True,
        limit=10000
    )

    coord_to_chunk = {}
    max_chunk_index_per_row = {}

    for point in points:
        coord = point.payload.get("coordinate")
        chunk = point.payload.get("chunk")

        if isinstance(coord, list) and len(coord) == 2:
            row_id, chunk_id = coord
            coord_tuple = (row_id, chunk_id)
            coord_to_chunk[coord_tuple] = chunk

            # Track max chunk index per row
            if row_id not in max_chunk_index_per_row:
                max_chunk_index_per_row[row_id] = chunk_id
            else:
                max_chunk_index_per_row[row_id] = max(max_chunk_index_per_row[row_id], chunk_id)

    print(f"🗺 Total chunked coords loaded: {len(coord_to_chunk)}")
    print(f"🗂 Example row max indices: {list(max_chunk_index_per_row.items())[:5]}")

    print("🧵 Step 3: Stitch neighbors")

    joined_results = []

    for (row_id, chunk_id) in primary_coords:
        max_chunk_id = max_chunk_index_per_row.get(row_id, chunk_id)

        parts = []

        if chunk_id > 0:
            prev = coord_to_chunk.get((row_id, chunk_id - 1))
            if prev:
                parts.append(prev)

        main = coord_to_chunk.get((row_id, chunk_id))
        if main:
            parts.append(main)

        if chunk_id < max_chunk_id:
            next_chunk = coord_to_chunk.get((row_id, chunk_id + 1))
            if next_chunk:
                parts.append(next_chunk)

        stitched_text = "\n".join(parts).strip()

        joined_results.append({
            "coordinate": [row_id, chunk_id],
            "joined_chunk": stitched_text
        })

    print(f"✅ Generated {len(joined_results)} stitched results.")
    return joined_results


In [14]:
def search_with_hypothetical_embedding(hypothetical_text, q_client, top_k_for_results=5):
    print("\n🔄 Step 3: Final Search using HyDE Embedding with Neighbor Context")

    # Step 1: Embed the hypothetical text
    hyde_embedding = embedding_model.encode(hypothetical_text, convert_to_tensor=True)

    # Step 2: Search with neighbors
    hyde_results = search_vector_with_neighbors(
        q_client,
        embedded_vector=hyde_embedding,
        collection_name="test_novel2",
        top_k=top_k_for_results
    )

    # Step 3: Print results
    for i, result in enumerate(hyde_results):
        print(f"\n[HyDE] Result {i+1} (Center Coord: {result['coordinate']}):")
        print(result['joined_chunk'])

    return hyde_results



In [15]:
query = "神裂火织都展现了甚么能力"

In [16]:
# Step 1: Generate HyDE prompt using top 2 context chunks
hypothetical_text = generate_hypothetical_query(query, q_client, top_k_for_context=3)

🔍 Step 1: Retrieve Initial Chunks for Hypothetical Generation
Performing search...

Initial Chunk 1:
。
圣人再度彼此交锋。
速度之快，让人根本来不及确认如此简单的事实。
神裂火织高速挥舞着七天七刀，趁机使出七根钢丝，一有时间就将刀入鞘，再次使出速度极快的拔刀术。她还利用了钢丝画出的三次元魔法阵与自身的脚步，以及钢铁与钢铁互击的韵律，制造出火焰与水的攻击术式，不断重复着这样的奇袭。
相较之下，水用巨大的铁棒弹开敌人的刀，并利用「神之力」的属性，吸取含有月光的夜气，增加铁棒的破坏力。另外他靠着圣母崇拜「对严惩的缓和」的特性，克服了「『神之右席』无法使用普通魔法」的条件。在放出超音速的连续攻击同时，使用真空刃与岩块从各角度攻击神裂。
咚嘎嘎嘎唰唰唰吱吱吱！火花四散。
神裂与水的周围，出现小规模的星空。
「唔…呼？」
但是，结果一目了然。
已经到达极限的神裂，口中流出断续的鲜血。她身体看不到的地方应该受了重伤。挥刀的速度明额变慢，甚至让人担心她最后可能会跟不上速度，受到水的一击。她根本无法朝好不
Coordinate: [15, 191]

Initial Chunk 2:
盖失去力气，手扶在墙壁上。这些人都露出同样的表情。
压倒性的无力感。
自己到底在做什么？五和心想。
神裂火织越是为自己而战，她就觉得自己的努力被否定。不管再怎么努力，自己也无法逃离圣人的手掌，「她」以充满关爱的眼神看着这种情景，在危险突然迫近时，自己一个人展开高层次的战斗。
她从没有正视过自己。
不管到哪里，自己跟伙伴们做的事都只像在玩家家酒。
被迫面对这样残酷的事实，还有神裂火织拚命表现出的温柔，心里却只能想着这种事，自己实在太过渺小，这使得五和他们的心灵大受打击。没办法，自己真的太渺小了。光是看着自己根本无法插手的压倒性战斗，强烈的无力感已经完全夺走受伤的身体内，残留的仅有体力与力量。
那个少年如果在这里，会在意这种事吗？
光是看到神裂火织这个「伙伴」在眼前战斗，被敌人伤害，他应该会马上握着拳头进入激烈的战斗中心吧。
这也是一种坚强的表现。
但是，现在的天草式却做不到这种坚强。
圣人之
Coordinate: [15, 195]

Initial Chunk 3:
神裂就会自灭。在「神之力」那忽缓忽

In [17]:
# Step 2: Perform final retrieval using HyDE embedding with top 5 results
final_results = search_with_hypothetical_embedding(hypothetical_text, q_client, top_k_for_results=5)


🔄 Step 3: Final Search using HyDE Embedding with Neighbor Context
🔍 Step 1: Vector Search
📎 Raw coordinate: [3, 241]
📎 Raw coordinate: [3, 242]
📎 Raw coordinate: [15, 191]
📎 Raw coordinate: [3, 243]
📎 Raw coordinate: [15, 216]
✅ Primary coords: [(3, 241), (3, 242), (15, 191), (3, 243), (15, 216)]
📌 All needed coords (with neighbors): [(3, 240), (3, 241), (3, 242), (3, 243), (3, 244), (15, 190), (15, 191), (15, 192), (15, 215), (15, 216), (15, 217)]
📦 Step 2: Fetch all points from Qdrant
🗺 Total chunked coords loaded: 6659
🗂 Example row max indices: [(0, 301), (1, 324), (2, 295), (3, 304), (4, 327)]
🧵 Step 3: Stitch neighbors
✅ Generated 5 stitched results.

[HyDE] Result 1 (Center Coord: [3, 241]):
神之力」，「神之力」却可以攻击到神裂。
而且在天使的快速连击之下，神裂甚至没有时间将拔出的长刀收回刀鞘中。无法使用擅长的拔刀术，神裂只好拚命挥动长刀防御。任谁来看，都知道神裂处于劣势。
神裂紧咬牙关忍耐。
伦敦排名前十强的魔法师，神裂。
在神裂火织的人生之中，一对一的情况下败北的次数，少得用两只手的手指就可以数得出来。而且所谓的「一对一」不见得是「人对人」，有时是「人对兽王」，甚至是「人对兵器」。
但是，这样的纪录如今似乎将面临重大考验。
原本用两手手指就可以数得完的「纪录」，似乎将变得无法数完了。
只不过，以这种超越常理的天使为对手，是否应该被列入纪录之中，本身就是一个大问